# [Project title]

**Team members:** [Names]  
**Repository:** [URL of your team's fork]

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "The data directory is missing."

## Load data

In [ ]:
# Load data from DATA_DIR.
# load flu data
flu_df = pd.read_csv(DATA_DIR / "target-hospital-admissions.csv")

# inspect the data
display(flu_df.head())
    # note: preseve location as a string ( DON'T Perform math on it)
display(flu_df.info())
    # note: date, location, location_name -> strings
    # note: value, weekly_rate -> float
print("missing values in date: ", flu_df['date'].isnull().sum())
print("missing values in location: ", flu_df['location'].isnull().sum())
print("missing values in location_name: ", flu_df['location_name'].isnull().sum())
print("missing values in value: ", flu_df['value'].isnull().sum())
print("missing values in weekly_rate: ", flu_df['weekly_rate'].isnull().sum())
    # note: missing values in value ( 36) and missing values in weekly_rate ( 36)

print("Earliest date:\n ", flu_df["date"].min())
    # note: february 5, 2022
print("Latest date: \n", flu_df["date"].max())
    # note: october 31st, 2026

print("location names: \n",flu_df['location_name'].unique())
print("location names count: \n", flu_df['location_name'].nunique())
    # note: 53
    # discard: District of Columbia, US, and Puerto rico
    # keep : 50 states ( since focusing on the states in the U.S)

print("number of duplicate rows: \n", flu_df.duplicated().sum())
    # note: 0
print("number of duplicate date and location combination: \n",flu_df.duplicated(subset=["date", "location"]).sum())
    # note: 0 

# display the locations with missing hospitalization counts
print("locations with missing hospitalization counts:\n",flu_df[flu_df['value'].isna()]['location_name'].value_counts())
    # note: massachusetts, west virginia, minesota
# look at missing hospitalizations and the date, location_name, value, and weekly rate associated with it
print(flu_df[flu_df["value"].isna()][
    ["date", "location_name", "value", "weekly_rate"]

])
    # note: value and weekly_rate is missing for both
    # missing values occured around the same time as well ( ex: october 5, september 28 21 14 7, 
    # august 31 24 17 10 3, july 27, june 22 15 08 01,  may 25 18 all in 2024 )

# how many unique dates each locatation has
print(flu_df.groupby("location_name")["date"].nunique().describe())
    # each fo the 53 states has 248 dates

,date,location,location_name,value,weekly_rate
0,2026-10-31,37,North Carolina,202.0,1.828712
1,2026-10-31,42,Pennsylvania,557.0,4.258816
2,2026-10-31,04,Arizona,138.0,1.820008
3,2026-10-31,54,West Virginia,23.0,1.299450
4,2026-10-31,11,District of Columbia,10.0,1.423994


<class 'pandas.DataFrame'>
RangeIndex: 13144 entries, 0 to 13143
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           13144 non-null  str    
 1   location       13144 non-null  str    
 2   location_name  13144 non-null  str    
 3   value          13108 non-null  float64
 4   weekly_rate    13108 non-null  float64
dtypes: float64(2), str(3)
memory usage: 777.8 KB


None

missing values in date:  0
missing values in location:  0
missing values in location_name:  0
missing values in value:  36
missing values in weekly_rate:  36
Earliest date:
  2022-02-05
Latest date: 
 2026-10-31
location names: 
 <ArrowStringArray>
[      'North Carolina',         'Pennsylvania',              'Arizona',
        'West Virginia', 'District of Columbia',        'New Hampshire',
               'Nevada',         'Rhode Island',               'Alaska',
       'South Carolina',                 'Iowa',             'Maryland',
             'Colorado',             'Delaware',                'Maine',
                 'Ohio',              'Georgia',            'Minnesota',
            'Wisconsin',             'Oklahoma',              'Wyoming',
          'Mississippi',             'New York',          'Connecticut',
              'Indiana',        'Massachusetts',           'Washington',
              'Alabama',          'Puerto Rico',               'Hawaii',
               'Kansa

## Prepare data

In [56]:
# Clean data and create features.

# cleaning data
    # steps to do:
    # convert to date time
    # check if weekly date interval
    # handle missing values
    # filter out non-U.S states

# convert date to date time
flu_df["date"] = pd.to_datetime(flu_df["date"])

date_gaps = (
    flu_df.sort_values(["location", "date"])
    .groupby("location")["date"]
    .diff()
    .dt.days
)

print(date_gaps.value_counts().sort_index())
    # note: all 13,091 date intervals are 7 days apart

print("zero hospitilization count: ", (flu_df["value"] == 0).sum())
    # note: 1054
# check if any negative hospitilizations
print("negative hospitilization counts: ", (flu_df["value"]< 0).sum())
    # note: 0
display(flu_df["value"].describe())
    #13,108 non missing hospitilization values
    # mean = 224.60 average weekly count across locations
    # median = 20
        # could be right-skewed
    # max = largest weekly count 

# clean the data
flu_clean = flu_df.copy()
print(flu_clean.isna().sum())

    # leave missing rows since imputing missing to be unknown would
    # change the entire value and weekly_rate column to be a str rather than a numeric value

# convert to datetime
flu_clean["date"] = pd.to_datetime(flu_clean["date"])
print(flu_clean["date"].dtype)
print(flu_clean.dtypes)

# strip any white spaces
flu_clean["location_name"] = flu_clean["location_name"].str.strip()
flu_clean["location"] = flu_clean["location"].str.strip()

# only keep the 50 States in the US
sorted(flu_clean["location_name"].unique())

exclude_locations = ["District of Columbia", "Puerto Rico", "US"]
flu_clean = flu_clean.drop(
    flu_clean[flu_clean["location_name"].isin(exclude_locations)].index
)
# check
print("Rows:", len(flu_clean))
print("Location:", flu_clean["location_name"].nunique())

# save cleaned csv
flu_clean.to_csv(DATA_DIR / "flu_cleaned.csv", index=False)

date
7.0    13091
Name: count, dtype: int64
zero hospitilization count:  1054
negative hospitilization counts:  0


count    13108.000000
mean       224.596277
std       1585.864307
min          0.000000
25%          5.000000
50%         20.000000
75%         88.000000
max      55544.000000
Name: value, dtype: float64

date              0
location          0
location_name     0
value            36
weekly_rate      36
dtype: int64
datetime64[us]
date             datetime64[us]
location                    str
location_name               str
value                   float64
weekly_rate             float64
dtype: object
Rows: 12400
Location: 50


### Feature engineering and chronological splitting

The prediction target is the number of influenza hospital admissions
in a state during the target week. Predictors are constructed using
hospitalization observations from preceding weeks, seasonal information,
and state identity.

Missing observations are not treated as zero. Rows without sufficient
historical observations are excluded from modeling.

Training, validation, and test sets are separated chronologically
to prevent future observations from entering model training.


In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned state-level dataset
panel = pd.read_csv(
    DATA_DIR / "flu_cleaned.csv",
    dtype={"location": "string"},
    parse_dates=["date"]
)

panel = panel.rename(columns={
    "location": "state",
    "location_name": "state_name",
    "date": "week",
    "value": "hospitalizations"
})

panel = panel.sort_values(
    ["state", "week"]
).reset_index(drop=True)

# Validate geographic identifiers and weekly observations
assert panel["state"].str.fullmatch(r"[0-9]{2}").all()
assert not panel.duplicated(["state", "week"]).any()
assert panel["state"].nunique() == 50

gaps = panel.groupby("state")["week"].diff().dropna()
assert gaps.eq(pd.Timedelta(days=7)).all()

# Construct lag features separately for each state
grouped = panel.groupby("state")["hospitalizations"]

for lag in range(1, 5):
    panel[f"lag_{lag}"] = grouped.shift(lag)

lag_columns = [f"lag_{i}" for i in range(1, 5)]

# Historical averages and changes
panel["rolling_mean_4"] = panel[lag_columns].mean(
    axis=1, skipna=False
)

panel["weekly_change"] = panel["lag_1"] - panel["lag_2"]

# Cyclical seasonal features
day = panel["week"].dt.dayofyear - 1
panel["season_sin"] = np.sin(2 * np.pi * day / 365.25)
panel["season_cos"] = np.cos(2 * np.pi * day / 365.25)

# Exclude incomplete historical windows and missing targets
required = lag_columns + ["hospitalizations"]
model_df = panel.dropna(subset=required).copy()

# Exclude the incomplete 2026-27 season
model_df = model_df[
    model_df["week"] < pd.Timestamp("2026-10-01")
].copy()

# Chronological splits based on the target week
model_df["split"] = np.select(
    [
        model_df["week"] < pd.Timestamp("2024-10-01"),
        model_df["week"] < pd.Timestamp("2025-10-01")
    ],
    ["train", "validation"],
    default="test"
)

train_df = model_df[model_df["split"] == "train"].copy()
validation_df = model_df[
    model_df["split"] == "validation"
].copy()
test_df = model_df[model_df["split"] == "test"].copy()

# Confirm chronological separation
assert train_df["week"].max() < validation_df["week"].min()
assert validation_df["week"].max() < test_df["week"].min()
assert not model_df[required].isna().any().any()

print("Original cleaned observations:", len(panel))
print("Modeling observations:", len(model_df))
print("Number of states:", model_df["state"].nunique())

print("\nSplit summary:")
print(model_df.groupby("split").agg(
    observations=("week", "size"),
    start_date=("week", "min"),
    end_date=("week", "max"),
    states=("state", "nunique")
).to_string())

print("\nExample feature rows:")
print(model_df[[
    "state", "week", "hospitalizations",
    "lag_1", "lag_2", "lag_3", "lag_4",
    "rolling_mean_4", "weekly_change"
]].head(10).to_string(index=False))

print("\nFeature engineering checks passed.")

## Explore data

In [ ]:
# Summarize and visualize the prepared data.

## Build models

### Persistence baseline and linear regression

The persistence baseline predicts that hospitalizations in the target
week will equal hospitalizations in the preceding week.

Four linear regression feature configurations are compared using
validation MAE. The configuration with the lowest validation MAE
is selected before examining test performance.

After selection, the final model is refitted using training and
validation data, then applied to the held-out test period.

Negative hospitalization predictions are clipped to zero because
hospitalization counts cannot be negative.

Training and validation predictions come from the selected model
fitted on training data only. Test predictions come from the
refitted model.

The evaluation uses finalized historical CDC records rather than
real-time reporting snapshots, which may introduce revision-related
limitations.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

target = "hospitalizations"

# Candidate feature configurations
feature_sets = {
    "Lag 1": ["lag_1"],
    "Lags 1-4": lag_columns,
    "Lags + seasonality": lag_columns + [
        "season_sin", "season_cos"
    ],
    "Lags + seasonality + state": lag_columns + [
        "season_sin", "season_cos", "state"
    ]
}

def build_regression(features):
    numeric = [f for f in features if f != "state"]

    transformers = [
        ("numeric", "passthrough", numeric)
    ]

    if "state" in features:
        transformers.append((
            "state",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            ["state"]
        ))

    return Pipeline([
        ("preprocessing", ColumnTransformer(transformers)),
        ("regression", LinearRegression())
    ])

def predict_counts(model, frame, features):
    return np.maximum(
        model.predict(frame[features]), 0
    )

def calculate_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        )
    }

# Persistence baseline on validation data
baseline_metrics = calculate_metrics(
    validation_df[target],
    validation_df["lag_1"]
)

validation_results = []
candidate_models = {}

# Fit candidates using training data only
for name, features in feature_sets.items():
    model = build_regression(features)

    model.fit(
        train_df[features],
        train_df[target]
    )

    val_pred = predict_counts(
        model, validation_df, features
    )

    metrics = calculate_metrics(
        validation_df[target], val_pred
    )

    validation_results.append({
        "configuration": name,
        **metrics
    })

    candidate_models[name] = model

validation_results = pd.DataFrame(
    validation_results
).sort_values("MAE", kind="stable")

# Select configuration without using test outcomes
best_name = validation_results.iloc[0]["configuration"]
best_features = feature_sets[best_name]
selected_model = candidate_models[best_name]

print("VALIDATION RESULTS")
print(
    validation_results.round(3).to_string(index=False)
)

print("\nPersistence validation metrics:")
print({
    k: round(v, 3)
    for k, v in baseline_metrics.items()
})

print("\nSelected configuration:", best_name)
print("Selected features:", best_features)

# Refit the selected configuration on all pre-test data
development_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

assert development_df["week"].max() < test_df["week"].min()

final_model = build_regression(best_features)

final_model.fit(
    development_df[best_features],
    development_df[target]
)

# Generate the prediction handoff
prediction_frames = []

for split_name, frame in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:
    model = (
        final_model if split_name == "test"
        else selected_model
    )

    result = frame[[
        "state", "week", target
    ]].copy()

    result = result.rename(
        columns={target: "actual"}
    )

    result["predicted_linear_regression"] = (
        predict_counts(model, frame, best_features)
    )

    result["predicted_baseline"] = (
        frame["lag_1"].to_numpy()
    )

    result["split"] = split_name

    prediction_frames.append(result)

predictions = pd.concat(
    prediction_frames, ignore_index=True
).sort_values(["week", "state"]).reset_index(drop=True)

# Validate prediction integrity
assert len(predictions) == len(model_df)
assert not predictions.duplicated(
    ["state", "week"]
).any()

prediction_columns = [
    "actual",
    "predicted_linear_regression",
    "predicted_baseline"
]

assert np.isfinite(
    predictions[prediction_columns].to_numpy()
).all()

assert predictions[prediction_columns].ge(0).all().all()

# Independently verify persistence against previous week
history = panel[[
    "state", "week", "hospitalizations"
]].copy()

history["week"] += pd.Timedelta(days=7)

history = history.rename(
    columns={"hospitalizations": "previous_actual"}
)

audit = predictions.merge(
    history,
    on=["state", "week"],
    validate="one_to_one"
)

assert len(audit) == len(predictions)
assert np.allclose(
    audit["predicted_baseline"],
    audit["previous_actual"]
)

# Save predictions for the evaluation section
predictions.to_csv(
    DATA_DIR / "predictions.csv",
    index=False
)

print("\nPrediction counts:")
print(predictions["split"].value_counts())

print("\nSample test predictions:")
print(
    predictions[predictions["split"] == "test"]
    .head(10).round(3).to_string(index=False)
)

print("\nAll prediction integrity checks passed.")
print("Saved data/predictions.csv")

## Evaluate models

In [ ]:
# Compare models using the chosen evaluation measures.

## Final results

In [ ]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]